# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

import numpy as np
import cv2
from sklearn.preprocessing import StandardScaler

import tkinter as tk

# --- Incarc MNIST si normalizez ---
transform = transforms.Compose([
    transforms.ToTensor(),  # transforma la [0,1], shape (1,28,28)
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

x_train = train_dataset.data.numpy()
y_train = train_dataset.targets.numpy()
x_test = test_dataset.data.numpy()
y_test = test_dataset.targets.numpy()

# Normalizez la [0,1]
x_train = x_train / 255.0
x_test = x_test / 255.0

In [2]:
# Functia pentru calcul geometric features
def calculate_geometric_features(image):
    image_uint8 = (image * 255).astype(np.uint8)
    moments = cv2.moments(image_uint8)

    area = moments['m00']
    cx = moments['m10'] / area if area != 0 else 0
    cy = moments['m01'] / area if area != 0 else 0

    Ixx = moments['mu20'] / area if area != 0 else 0
    Iyy = moments['mu02'] / area if area != 0 else 0
    Ixy = moments['mu11'] / area if area != 0 else 0

    angle = 0.5 * np.arctan2(2 * Ixy, Ixx - Iyy)

    contours, _ = cv2.findContours(image_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    perimeter = cv2.arcLength(contours[0], True) if contours else 0

    thinness = (4 * np.pi * area) / (perimeter ** 2) if perimeter != 0 else 0

    x, y, w, h = cv2.boundingRect(image_uint8)
    elongation = h / w if w != 0 else 0

    return [area, cx, cy, angle, perimeter, thinness, elongation]

In [3]:
# Calcul geometric features pentru seturi
geo_train = np.array([calculate_geometric_features(img) for img in x_train])
geo_test = np.array([calculate_geometric_features(img) for img in x_test])

# Scalez trăsături geometrice
scaler = StandardScaler()
geo_train = scaler.fit_transform(geo_train)
geo_test = scaler.transform(geo_test)

# Convertesc totul in torch tensors
x_train_tensor = torch.tensor(x_train).unsqueeze(1).float()  # (N, 1, 28, 28)
geo_train_tensor = torch.tensor(geo_train).float()
y_train_tensor = torch.tensor(y_train).long()

x_test_tensor = torch.tensor(x_test).unsqueeze(1).float()
geo_test_tensor = torch.tensor(geo_test).float()
y_test_tensor = torch.tensor(y_test).long()

# Creez DataLoader (opțional)
train_loader = DataLoader(torch.utils.data.TensorDataset(x_train_tensor, geo_train_tensor, y_train_tensor),
                          batch_size=64, shuffle=True)
test_loader = DataLoader(torch.utils.data.TensorDataset(x_test_tensor, geo_test_tensor, y_test_tensor),
                         batch_size=64, shuffle=False)

In [4]:
class CNNGeoModel(nn.Module):
    def __init__(self):
        super(CNNGeoModel, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)

        self.flatten_dim = 64 * 7 * 7  # dupa 2 pool-uri pe 28x28
        self.fc1 = nn.Linear(self.flatten_dim + 7, 128)  # 7 = nr trăsături geometrice
        self.fc2 = nn.Linear(128, 10)

    def forward(self, img, geo):
        x = F.relu(self.conv1(img))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(-1, self.flatten_dim)
        x = torch.cat([x, geo], dim=1)  # concatenare trăsături geometrice
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNGeoModel().to(device)
print(device)

cuda


In [5]:
# def train_model(model, train_loader, epochs=5):
#     criterion = nn.CrossEntropyLoss()
#     optimizer = optim.Adam(model.parameters())
# 
#     model.train()
#     for epoch in range(epochs):
#         running_loss = 0.0
#         correct = 0
#         total = 0
#         for imgs, geos, labels in train_loader:
#             imgs, geos, labels = imgs.to(device), geos.to(device), labels.to(device)
# 
#             optimizer.zero_grad()
#             outputs = model(imgs, geos)
#             loss = criterion(outputs, labels)
#             loss.backward()
#             optimizer.step()
# 
#             running_loss += loss.item()
#             _, predicted = torch.max(outputs.data, 1)
#             total += labels.size(0)
#             correct += (predicted == labels).sum().item()
# 
#         print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f} - Accuracy: {100*correct/total:.2f}%")
# 
# # Pentru antrenare, rulează:
# train_model(model, train_loader, epochs=5)
# 
# # Salvare model după antrenare
# torch.save(model.state_dict(), './models/m1.pth')


In [6]:
def test_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, geos, labels in test_loader:
            imgs, geos, labels = imgs.to(device), geos.to(device), labels.to(device)
            outputs = model(imgs, geos)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

# Pentru testare
model.load_state_dict(torch.load('./models/m1.pth'))
test_model(model, test_loader)


Test Accuracy: 99.06%


In [1]:
class DrawingApp:
    def __init__(self, root, model, scaler):
        self.root = root
        self.model = model
        self.scaler = scaler
        self.model.eval()

        self.root.title("Desenează o cifră")

        self.canvas = tk.Canvas(root, width=280, height=280, bg='white')
        self.canvas.pack()
        self.canvas.bind("<B1-Motion>", self.paint)

        self.image = np.zeros((280, 280), dtype=np.uint8)
        self.drawing = False

        self.predict_button = tk.Button(root, text="Recunoaște", command=self.recognize)
        self.predict_button.pack()

        self.clear_button = tk.Button(root, text="Șterge", command=self.clear)
        self.clear_button.pack()

        self.prediction_label = tk.Label(root, text="Cifra prezisă: -", font=("Helvetica", 16))
        self.prediction_label.pack()

        self.accuracy_label = tk.Label(root, text="Probabilitate: -", font=("Helvetica", 14))
        self.accuracy_label.pack()

    def paint(self, event):
        x, y = event.x, event.y
        r = 8
        self.canvas.create_oval(x - r, y - r, x + r, y + r, fill='black')
        if 0 <= x < 280 and 0 <= y < 280:
            self.image[y-r:y+r, x-r:x+r] = 255
            self.drawing = True

    def clear(self):
        self.canvas.delete("all")
        self.image.fill(0)
        self.prediction_label.config(text="Cifra prezisă: -")
        self.accuracy_label.config(text="Probabilitate: -")
        self.drawing = False

    def recognize(self):
        if not self.drawing:
            return

        # Redimensionez la 28x28 si normalizez
        img28 = cv2.resize(self.image, (28, 28))
        img28 = img28.astype(np.float32) / 255.0

        # Calculez trăsături geometrice si scalez
        geo = np.array(calculate_geometric_features(img28)).reshape(1, -1)
        geo_scaled = self.scaler.transform(geo)

        # Convert la tensor torch
        img_tensor = torch.tensor(img28).unsqueeze(0).unsqueeze(0).float().to(device)  # (1,1,28,28)
        geo_tensor = torch.tensor(geo_scaled).float().to(device)  # (1,7)

        with torch.no_grad():
            outputs = self.model(img_tensor, geo_tensor)
            probs = torch.softmax(outputs, dim=1)
            prob_val, pred = torch.max(probs, 1)

        self.prediction_label.config(text=f"Cifra prezisă: {pred.item()}")
        self.accuracy_label.config(text=f"Probabilitate: {prob_val.item()*100:.2f}%")
        self.drawing = False

# Pentru a porni aplicația:

root = tk.Tk()
app = DrawingApp(root, model, scaler)
root.mainloop()


NameError: name 'tk' is not defined

In [8]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.backends.cudnn.version())
print(torch.cuda.is_available())
print(torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.7.0+cu128
12.8
90701
True
1
NVIDIA GeForce RTX 3050 Laptop GPU
